In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

import re

In [2]:
PROJECT_DIR = Path.cwd().parent

RAW_DATA = PROJECT_DIR / "Extracted_Data"

PROCESSED = PROJECT_DIR / "processed"

clean_df = pd.read_csv(

    PROCESSED/"cleaned_dataset.csv"

)

print(clean_df.shape)

(23424, 48)


In [3]:
excel_files = sorted(

    RAW_DATA.rglob("*.xlsx")

)

len(excel_files)

24

In [4]:
def extract_date(filepath):

    temp = pd.read_excel(

        filepath,

        sheet_name="Gesamt-Kfz",

        header=None,

        nrows=5

    )

    value = str(temp.iloc[2,2])

    return value

In [5]:
sample = excel_files[0]

print(extract_date(sample))

Tue, 08. Oct 2024


In [6]:
def convert_date(text):

    months = {

        "Jan":"Jan",

        "Feb":"Feb",

        "Mär":"Mar",

        "Apr":"Apr",

        "Mai":"May",

        "Jun":"Jun",

        "Jul":"Jul",

        "Aug":"Aug",

        "Sep":"Sep",

        "Okt":"Oct",

        "Nov":"Nov",

        "Dez":"Dec"

    }

    for german,english in months.items():

        text=text.replace(german,english)

    text=text.replace("Mon,","")

    text=text.replace("Tue,","")

    text=text.replace("Wed,","")

    text=text.replace("Thu,","")

    text=text.replace("Fri,","")

    text=text.replace("Sat,","")

    text=text.replace("Sun,","")

    return pd.to_datetime(

        text.strip(),

        errors="coerce"

    )

In [7]:
convert_date(

    extract_date(sample)

)

Timestamp('2024-10-08 00:00:00')

In [8]:
records = []

for file in excel_files:

    date = convert_date(

        extract_date(file)

    )

    records.append({

        "Source_File":file.name,

        "Date":date

    })

date_df = pd.DataFrame(records)

date_df.head()

,Source_File,Date
0,2024-10-08_to_2024-10-13_UI_202403B0923.xlsx,2024-10-08
1,2024-10-08_to_2024-10-13_UI_202310B0614.xlsx,2024-10-08
2,2024-10-08_to_2024-10-13_UI_202403B0924.xlsx,2024-10-08
3,2024-10-08_to_2024-10-13_UI_202403B0908.xlsx,2024-10-08
4,2024-09-09_to_2024-09-15_UI_202403B0908.xlsx,2024-09-09


In [9]:
enhanced = clean_df.merge(

    date_df,

    on="Source_File",

    how="left"

)

In [10]:
enhanced["Weekday"] = enhanced["Date"].dt.day_name()

enhanced["Month"] = enhanced["Date"].dt.month_name()

enhanced["Year"] = enhanced["Date"].dt.year

enhanced["Day"] = enhanced["Date"].dt.day

enhanced["Week_Number"] = enhanced["Date"].dt.isocalendar().week

In [11]:
enhanced[

    [

        "Date",

        "Weekday",

        "Month",

        "Year"

    ]

].head()

,Date,Weekday,Month,Year
0,2024-10-08,Tuesday,October,2024
1,2024-10-08,Tuesday,October,2024
2,2024-10-08,Tuesday,October,2024
3,2024-10-08,Tuesday,October,2024
4,2024-10-08,Tuesday,October,2024


In [12]:
enhanced["Weekday"].value_counts()

Weekday
Monday       15552
Tuesday       7008
Wednesday      864
Name: count, dtype: int64

In [14]:
enhanced.to_csv(

    PROCESSED/"enhanced_dataset.csv",

    index=False

)

print("Enhanced Dataset Saved")

Enhanced Dataset Saved


In [15]:
enhanced.head()

,Week,Measurement_Site,Latitude,Longitude,Source_File,Vehicle_Category,Time,von Unten nach Oben,von Links nach Oben,von Links nach Unten,...,Hour,Minute,Peak_Period,Time_Interval,Date,Weekday,Month,Year,Day,Week_Number
0,DZwEI 08.10-13.10,Glauburgstraße,50.127065,8.689309,2024-10-08_to_2024-10-13_UI_202403B0923.xlsx,Bus,1900-01-01 00:00:00,0.0,0.0,0.0,...,0,0,Off Peak,00:00,2024-10-08,Tuesday,October,2024,8,41
1,DZwEI 08.10-13.10,Glauburgstraße,50.127065,8.689309,2024-10-08_to_2024-10-13_UI_202403B0923.xlsx,Bus,1900-01-01 00:15:00,0.0,0.0,0.0,...,0,15,Off Peak,00:15,2024-10-08,Tuesday,October,2024,8,41
2,DZwEI 08.10-13.10,Glauburgstraße,50.127065,8.689309,2024-10-08_to_2024-10-13_UI_202403B0923.xlsx,Bus,1900-01-01 00:30:00,0.0,0.0,0.0,...,0,30,Off Peak,00:30,2024-10-08,Tuesday,October,2024,8,41
3,DZwEI 08.10-13.10,Glauburgstraße,50.127065,8.689309,2024-10-08_to_2024-10-13_UI_202403B0923.xlsx,Bus,1900-01-01 00:45:00,1.0,0.0,0.0,...,0,45,Off Peak,00:45,2024-10-08,Tuesday,October,2024,8,41
4,DZwEI 08.10-13.10,Glauburgstraße,50.127065,8.689309,2024-10-08_to_2024-10-13_UI_202403B0923.xlsx,Bus,1900-01-01 01:00:00,2.0,0.0,0.0,...,1,0,Off Peak,01:00,2024-10-08,Tuesday,October,2024,8,41


In [16]:
date_df.sort_values("Date")

,Source_File,Date
4,2024-09-09_to_2024-09-15_UI_202403B0908.xlsx,2024-09-09
5,2024-09-09_to_2024-09-15_UI_202403B0924.xlsx,2024-09-09
6,2024-09-09_to_2024-09-15_UI_202403B0923.xlsx,2024-09-09
7,2024-09-09_to_2024-09-15_UI_202310B0614.xlsx,2024-09-09
12,2024-09-16_to_2024-09-23_UI_202403B0908.xlsx,2024-09-16
13,2024-09-16_to_2024-09-23_UI_202403B0924.xlsx,2024-09-16
14,2024-09-16_to_2024-09-23_UI_202403B0923.xlsx,2024-09-16
15,2024-09-16_to_2024-09-23_UI_202310B0614.xlsx,2024-09-16
19,2024-09-24_to_2024-09-29_UI_202310B0614.xlsx,2024-09-24
18,2024-09-24_to_2024-09-29_UI_202403B0923.xlsx,2024-09-24


In [17]:
date_df["Date"].dt.day_name().value_counts()

Date
Monday       16
Tuesday       7
Wednesday     1
Name: count, dtype: int64

In [18]:
sample = excel_files[0]

raw = pd.read_excel(
    sample,
    sheet_name="Gesamt-Kfz",
    header=None
)

print(raw.iloc[0:120, 0:8])

                     0    1                                     2    3    4  \
0    Name der Erhebung  NaN                        UI_202403B0923  NaN  NaN   
1       Geräte-UUID(s)  NaN  d440422d-f10c-4884-905e-cc168d263334  NaN  NaN   
2            Zähldatum  NaN                     Tue, 08. Oct 2024  NaN  NaN   
3            Startzeit  NaN                                 00:00  NaN  NaN   
4             Standort  NaN                        (50.127,8.689)  NaN  NaN   
..                 ...  ...                                   ...  ...  ...   
115           01:45:00    5                                     0    0    1   
116           02:00:00    6                                     0    0    0   
117           02:15:00    1                                     0    1    2   
118           02:30:00    1                                     0    0    0   
119           02:45:00    1                                     0    0    1   

      5   6    7  
0   NaN NaN  NaN  
1   NaN NaN  